# 2D Decision Boundary

Train a classifier on **2D blob** data and visualize the decision regions.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)

class BlobDataGenerator:
    def __init__(self, n_per_class=80, seed=42):
        g = torch.Generator().manual_seed(seed)
        centers = torch.tensor([[0., 0.], [2., 2.], [-2., 1.]])
        X, y = [], []
        for c, center in enumerate(centers):
            pts = center + torch.randn(n_per_class, 2, generator=g) * 0.6
            X.append(pts)
            y.append(torch.full((n_per_class,), c))
        self.X = torch.cat(X)
        self.y = torch.cat(y)

class BlobDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

class BlobClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 3))
    def forward(self, x):
        return self.net(x)


## 1. Generate 2D blobs

In [ ]:
gen = BlobDataGenerator()
X, y = gen.X, gen.y
print(X.shape, y.shape)

## 2. Scatter plot of classes

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71']
for c in range(3):
    mask = y == c
    ax.scatter(X[mask, 0], X[mask, 1], c=colors[c], label=f'class {c}', s=30, alpha=0.8)
ax.legend(); ax.set_title('2D blob dataset'); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

## 3. Train classifier

In [ ]:
model = BlobClassifier()
opt = torch.optim.Adam(model.parameters(), lr=0.05)
for _ in range(200):
    opt.zero_grad()
    F.cross_entropy(model(X), y).backward()
    opt.step()

## 4. Decision boundary mesh

In [ ]:
xx, yy = np.meshgrid(np.linspace(-4, 4, 120), np.linspace(-3, 4, 120))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad():
    preds = model(grid).argmax(1).numpy().reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, preds, alpha=0.35, cmap='Pastel1')
for c in range(3):
    mask = y == c
    ax.scatter(X[mask, 0], X[mask, 1], c=colors[c], s=25, edgecolors='k', linewidths=0.3)
ax.set_title('Decision regions + data points'); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

## 5. Confidence heatmap

In [ ]:
with torch.no_grad():
    probs = F.softmax(model(grid), dim=1)[:, 0].numpy().reshape(xx.shape)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(probs, extent=[-4, 4, -3, 4], origin='lower', cmap='viridis', alpha=0.9)
plt.colorbar(im, label='P(class 0)')
ax.scatter(X[y==0, 0], X[y==0, 1], c='white', s=15, edgecolors='k')
ax.set_title('Class-0 probability surface')
plt.tight_layout(); plt.show()